# Merge Dengue × Clima
Agrega dados históricos (`climate.csv`) e previsões climáticas (`forecasting_climate.csv`) ao nível UF e une com `dengue_uf_preprocessed.csv`.

In [ ]:
import pandas as pd
import numpy as np

## 1. Carregamento

In [ ]:
dengue = pd.read_csv('dengue_uf_preprocessed.csv', parse_dates=['date'])

# climate.csv é grande (~7.6M linhas) — carrega com dtypes explícitos para economizar memória
climate = pd.read_csv(
    '../DataRaw/climate.csv',
    parse_dates=['date'],
    dtype={'geocode': 'int32', 'epiweek': 'int32', 'rainy_days': 'int8'}
)

forecast = pd.read_csv(
    '../DataRaw/forecasting_climate.csv',
    parse_dates=['reference_month'],
    dtype={'geocode': 'int32', 'forecast_months_ahead': 'int8'}
)

print('dengue:  ', dengue.shape)
print('climate: ', climate.shape)
print('forecast:', forecast.shape)

## 2. Mapeamento geocode → uf_code
No IBGE, os 2 primeiros dígitos do geocode identificam o estado.

In [ ]:
UF_CODE_ES = 32  # Espírito Santo

# Tabela uf_code → sigla derivada do próprio dengue (já processado)
uf_map = dengue[['uf_code', 'uf']].drop_duplicates().set_index('uf_code')['uf']

def add_uf_code(df):
    df = df.copy()
    df['uf_code'] = (df['geocode'] // 100_000).astype('int16')
    df = df[df['uf_code'] != UF_CODE_ES]
    df['uf'] = df['uf_code'].map(uf_map)
    return df

climate  = add_uf_code(climate)
forecast = add_uf_code(forecast)

print('Após exclusão ES — climate:', climate.shape, '| forecast:', forecast.shape)

## 3. Agregação de clima histórico por UF + semana epidemiológica

In [ ]:
CLIMATE_VARS = [
    'temp_min', 'temp_med', 'temp_max',
    'precip_min', 'precip_med', 'precip_max',
    'pressure_min', 'pressure_med', 'pressure_max',
    'rel_humid_min', 'rel_humid_med', 'rel_humid_max',
    'thermal_range', 'rainy_days'
]

climate_uf = (
    climate
    .groupby(['date', 'epiweek', 'uf_code', 'uf'], as_index=False)[CLIMATE_VARS]
    .mean()
)

print('climate_uf:', climate_uf.shape)
climate_uf.head()

## 4. Merge dengue × clima histórico

In [ ]:
df = dengue.merge(climate_uf, on=['date', 'epiweek', 'uf_code', 'uf'], how='left')

missing = df[CLIMATE_VARS].isna().mean().mul(100).round(1)
print('% missing por variável climática:')
print(missing[missing > 0])
print('\nShape após merge clima histórico:', df.shape)

## 5. Agregação de previsões climáticas por UF + mês de referência
Pivoteia `forecast_months_ahead` para colunas wide: `temp_med_1m`, `temp_med_2m`, …

In [ ]:
FORECAST_VARS = ['temp_med', 'umid_med', 'precip_tot']

forecast_uf = (
    forecast
    .groupby(['reference_month', 'forecast_months_ahead', 'uf_code', 'uf'], as_index=False)[FORECAST_VARS]
    .mean()
)

# Pivot: uma coluna por variável × mês de antecedência
forecast_wide = forecast_uf.pivot_table(
    index=['reference_month', 'uf_code', 'uf'],
    columns='forecast_months_ahead',
    values=FORECAST_VARS
)
forecast_wide.columns = [f'{var}_{n}m' for var, n in forecast_wide.columns]
forecast_wide = forecast_wide.reset_index()

print('forecast_wide:', forecast_wide.shape)
print('Colunas:', forecast_wide.columns.tolist())

## 6. Merge com previsões climáticas (chave: ano-mês × UF)

In [ ]:
# Alinha o mês de referência: início do mês da data de cada observação de dengue
df['reference_month'] = df['date'].values.astype('datetime64[M]')

df = df.merge(forecast_wide, on=['reference_month', 'uf_code', 'uf'], how='left')

forecast_cols = [c for c in df.columns if c.endswith('m') and any(v in c for v in FORECAST_VARS)]
missing_fc = df[forecast_cols].isna().mean().mul(100).round(1)
print('% missing nas previsões (amostra):')
print(missing_fc.head(10))
print('\nShape final:', df.shape)

## 7. Verificação e exportação

In [ ]:
print('Colunas finais:')
print(df.dtypes.to_string())
df.head()

In [ ]:
df.to_csv('dengue_uf_climate.csv', index=False)
print('Salvo em: 1_Preprocessing/dengue_uf_climate.csv')